# To-do list
* `nse.py` programs   
* [x] Get NSE symbols   
* [x] Build option chains   
* [x] Get accurate dte
* [x] Get prices   
* [x] Get margins   

## Get nse symbols

In [1]:
# hack to change jupyter directory in notebooks for imports and handle asyncio
import os
from pathlib import Path
import pandas as pd
from ib_insync import util
if Path.cwd().parts[-1] == 'notebooks':
    root = Path.cwd().parent
else:
    root = Path.cwd()
os.chdir(root)

import nest_asyncio
nest_asyncio.apply()
util.startLoop()
pd.options.display.max_columns = None
pd.options.display.float_format = '{:,.2f}'.format # set float precision with comma

# Logger
import logging, logging.config
import yaml

# with open(root / 'config' / 'log.yml', 'r') as f:
#     log_config = yaml.safe_load(f.read())
#     logging.config.dictConfig(log_config)

logging.getLogger('ib_insync').setLevel(logging.ERROR)
log = logging.getLogger('ib_log')


# Build Option Chains

# Get accurate dte

# Get prices of underlyings

## Change str to float and date to xDt to date in get_prices

# Get the margins

In [ ]:
import pandas as pd

# Get the options
chains = root/'data'/'master'/'nse_chains.pkl'
df_ch = pd.read_pickle(chains)

# Get the latest underlying prices
from src.nse.nse import get_prices
df_prices = get_prices()

undPrice = df_ch.symbol.map(df_prices.set_index('symbol')['last'].to_dict())
df_ch = df_ch.assign(undPrice=undPrice)


In [ ]:
# df_margin.to_pickle(root / 'data' / 'master' / 'nse_margins.pkl')

In [44]:
# Get margins for all chains
df_margin = pd.read_pickle(root / 'data' / 'master' / 'nse_margins.pkl')

# .. remove unnecessary unds from chains
df_ch = df_ch[df_ch.symbol.isin(df_margin.symbol)]

# .. get options that have some price
df_m = df_ch[df_ch.lastPrice>0].reset_index(drop=True)

# .. qualify the chains
from ib_insync import Contract, MarketOrder, IB

# get ib_symbols
from src.nse.nse import get_nse_syms
df_syms = get_nse_syms()

# dictionary map
m_di = df_syms.set_index('symbol').ib_sym.to_dict()

# add ib_sym to df_m
df_m.insert(1, 'ib_sym', df_m.symbol.map(m_di))

contracts = [Contract(symbol=s, lastTradeDateOrContractMonth=e, strike=k, 
                      right=r, secType ='OPT', exchange='NSE', currency='INR') 
   for s, e, k, r
   in zip(df_m.ib_sym, 
          df_m.expiry.dt.strftime('%Y%m%d'), 
          df_m.strike, 
          df_m.right, )]

ib = IB() # instantiate ib


In [48]:
# Make dataframe of successfully qualified contracts
df_opt_cts = pd.DataFrame(results).assign(contract=results)
df_opt_cts.rename(columns={'lastTradeDateOrContractMonth': 'expiry', 'symbol': 'ib_sym'}, inplace=True)

# Prepare df_m for the join to qualified df_opts_cts
df_tgt = df_m.assign(expiry = df_m.expiry.dt.strftime('%Y%m%d'))

idx_cols = ['ib_sym', 'expiry', 'strike', 'right']

df_tgt = df_tgt.set_index(idx_cols).join(df_opt_cts.set_index(idx_cols)[['conId', 'contract']]).dropna(subset=['contract']).reset_index()
df_tgt = df_tgt.assign(order=[MarketOrder('SELL', lot) for lot in df_tgt.lot])


In [3]:
from src.support import marginsAsync
import asyncio
from ib_insync import IB

ib = IB()

df_tgt = pd.read_pickle(root / 'data' / 'master' / 'nse_opt_margins.pkl')
opt_margin_di = asyncio.run(marginsAsync(ib, df_tgt.contract, df_tgt.order))
df = pd.concat(opt_margin_di.values(), ignore_index=True, axis=0).dtypes
df

615040948 to 615040983: 100%|█████████████████████████| 1/1 [00:02<00:00,  2.82s/it]
